In [1]:
"""
This file only works for 3D slice by slice inference.
"""

import os
import sys

sys.path.append("../..")
#import opensimplex

#from torchvision.utils import save_image

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
import json
import nibabel as nib
from scipy import stats
import argparse
import multiprocessing

import pandas as pd
import utils.scores as scores

from utils.utils import *


from utils.utils import *

from scipy.ndimage import median_filter, binary_erosion, binary_dilation, binary_fill_holes



/home/fehrdelt/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


### works when the anomaly maps 20x have already been computed and saved, now we only compute the Final dice scores


To compute with kraken cpu multiprocessing

In [2]:
DEVICE_TYPE = "cuda:0"


ROOT_DIR = "/home/fehrdelt/bettik/"
#ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"



In [3]:
# for ADC large group 20x inference
SUB_EXPERIMENT_NAME = "exp_2_2"
GROUP = "large"
timesteps = 90
median_filter_size = 7
threshold = 0.04
erosion_dilation_iterations = 6
binary_fill_holes_param = 1

# for ADC medium group 20x inference
# SUB_EXPERIMENT_NAME = "exp_2_2"
# GROUP = "medium"
#timesteps = 70
#median_filter_size = 7
#threshold = 0.04
#erosion_dilation_iterations = 4
#binary_fill_holes_param = 0

# for FLAIR large group 20x inference
# SUB_EXPERIMENT_NAME = "exp_3_2"
# GROUP = "large"
# timesteps = 150
# median_filter_size = 7
# threshold = 0.06
# erosion_dilation_iterations = 2
# binary_fill_holes_param = 1

# for FLAIR medium group 20x inference
# SUB_EXPERIMENT_NAME = "exp_3_2"
# GROUP = "medium"
# timesteps = 130
# median_filter_size = 7
# threshold = 0.06
# erosion_dilation_iterations = 2
# binary_fill_holes_param = 1

In [4]:
with open(ROOT_DIR+f"AnoDiffExperiments/experiment_{SUB_EXPERIMENT_NAME[4:5]}/{SUB_EXPERIMENT_NAME}/config.json", "r") as f:
    config = json.load(f)

# "args" namespace compatible with define_instance() and dataset classes.
args = argparse.Namespace(**config)

In [5]:

anomaly_maps_dir = ROOT_DIR+f"datasets/anomaly_maps/{SUB_EXPERIMENT_NAME}/{GROUP}_20x_inference/"
masks_dir = ROOT_DIR+f"datasets/final_soop_dataset_small/masks_combined_registered/"

In [6]:
iou_scores = []
dice_scores = []
hausdorff_distances = []
precision_scores = []
recall_scores = []
f1_scores = []

In [7]:


def process_file(anomaly_map_file):
    anomaly_map = nib.load(os.path.join(anomaly_maps_dir, anomaly_map_file)).get_fdata()
    mask_path = os.path.join(masks_dir, anomaly_map_file.split('_')[0] + ".nii.gz")
    mask = nib.load(mask_path).get_fdata()

    final_anomaly_map = torch.from_numpy(anomaly_map).float().unsqueeze(0).unsqueeze(0)
    test_masks = torch.from_numpy(mask).float().unsqueeze(0).unsqueeze(0)
    test_masks = (test_masks > 0).float()

    # make the segmentation map with threshold
    ano_segmentation = final_anomaly_map > threshold

    # perform erosion if specified
    if erosion_dilation_iterations > 0:
        ano_segmentation_np = ano_segmentation.cpu().numpy()
        for b in range(ano_segmentation_np.shape[0]):
            ano_segmentation_np[b,0] = binary_erosion(ano_segmentation_np[b,0], iterations=erosion_dilation_iterations)
            ano_segmentation_np[b,0] = binary_dilation(ano_segmentation_np[b,0], iterations=erosion_dilation_iterations)
        ano_segmentation = torch.from_numpy(ano_segmentation_np)
    
    if binary_fill_holes_param == 1:
        ano_segmentation_np = ano_segmentation.cpu().numpy()
        for b in range(ano_segmentation_np.shape[0]):
            ano_segmentation_np[b,0] = binary_fill_holes(ano_segmentation_np[b,0])
        ano_segmentation = torch.from_numpy(ano_segmentation_np)

    iou_scores_batch, dice_scores_batch, hausdorff_distances_batch, precision_scores_batch, recall_scores_batch, f1_scores_batch = scores.compute_scores(ano_segmentation, test_masks)

    return iou_scores_batch, dice_scores_batch, hausdorff_distances_batch, precision_scores_batch, recall_scores_batch, f1_scores_batch

# Multiprocessing execution
if __name__ == '__main__':
    files = os.listdir(anomaly_maps_dir)
    num_cores = 48

    with multiprocessing.Pool(processes=num_cores) as pool:
        results = list(tqdm(pool.imap(process_file, files), total=len(files)))

    # unpack and collect results
    for result in results:
        iou_scores.append(result[0])
        dice_scores.append(result[1])
        hausdorff_distances.append(result[2])
        precision_scores.append(result[3])
        recall_scores.append(result[4])
        f1_scores.append(result[5])


  0%|                                                                                                                                                                                                             | 0/48 [00:00<?, ?it/s]monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
monai.me

In [8]:
hausdorff_distances = [[x for x in sublist if not np.isnan(x)] for sublist in hausdorff_distances]

In [9]:

# Flatten nested lists before converting to numpy array
flat_iou = [item for sublist in iou_scores for item in sublist]
flat_dice = [item for sublist in dice_scores for item in sublist]
flat_hausdorff = [item for sublist in hausdorff_distances for item in sublist]
flat_precision = [item for sublist in precision_scores for item in sublist]
flat_recall = [item for sublist in recall_scores for item in sublist]
flat_f1 = [item for sublist in f1_scores for item in sublist]


mean_iou, lower_iou, upper_iou = scores.make_confidence_intervals(np.array(flat_iou))

mean_dice, lower_dice, upper_dice = scores.make_confidence_intervals(np.array(flat_dice))

mean_hausdorff, lower_hausdorff, upper_hausdorff = scores.make_confidence_intervals(np.array(flat_hausdorff))

mean_precision, lower_precision, upper_precision = scores.make_confidence_intervals(np.array(flat_precision))

mean_recall, lower_recall, upper_recall = scores.make_confidence_intervals(np.array(flat_recall))

mean_f1, lower_f1, upper_f1 = scores.make_confidence_intervals(np.array(flat_f1))

final_scores = {
    "iou": [round(mean_iou, 4), round(lower_iou, 4), round(upper_iou, 4)],
    "dice": [round(mean_dice, 4), round(lower_dice, 4), round(upper_dice, 4)],
    "hausdorff": [round(mean_hausdorff, 4), round(lower_hausdorff, 4), round(upper_hausdorff, 4)],
    "precision": [round(mean_precision, 4), round(lower_precision, 4), round(upper_precision, 4)],
    "recall": [round(mean_recall, 4), round(lower_recall, 4), round(upper_recall, 4)],
    "f1": [round(mean_f1, 4), round(lower_f1, 4), round(upper_f1, 4)]
}


invalid value encountered in subtract


In [10]:
print("for experiment:", SUB_EXPERIMENT_NAME)
# print the final scores with confidence intervals
print("Final Scores with 95% Confidence Intervals:")
for metric, value in final_scores.items():
    print(f"{metric.capitalize()}: {np.round(value[0],4)} ({np.round(value[1],4)} - {np.round(value[2],4)})")

for experiment: exp_2_2
Final Scores with 95% Confidence Intervals:
Iou: 0.2953999936580658 (0.2233 - 0.3683)
Dice: 0.39640000462532043 (0.3084 - 0.4795)
Hausdorff: inf (nan - nan)
Precision: 0.7792 (0.687 - 0.878)
Recall: 0.3102 (0.2344 - 0.3891)
F1: 0.3964 (0.3084 - 0.4795)
